# 3.3 LCEL — LangChain Expression Language

## Playground Notebook

LCEL is LangChain's **declarative composition framework**. It lets you build complex LLM pipelines by chaining components with the pipe `|` operator.

| Topic | What You'll Learn |
|-------|-------------------|
| **The Pipe Operator** | Building chains with `prompt \| model \| parser` |
| **Runnable Protocol** | `invoke()`, `stream()`, `batch()` — the universal interface |
| **RunnablePassthrough** | Forwarding and enriching data through chains |
| **RunnableParallel** | Executing multiple branches simultaneously |
| **RunnableLambda** | Injecting custom Python functions into chains |
| **RunnableBranch** | Conditional routing (if/else for chains) |
| **Fallbacks & Retry** | Automatic error recovery and resilience |
| **Streaming & Batching** | First-class streaming and parallel batch processing |

Think of LCEL as the **plumbing system** — individual components are useful on their own, but LCEL connects them into a pipeline where data flows seamlessly and streaming, async, batching, and observability come for free.

> **Model:** `llama3.2:3b` via **Ollama** — running 100% locally, no API keys needed.

---

In [ ]:
# ============================================================
#  INSTALL DEPENDENCIES (run once)
# ============================================================
# !pip install langchain langchain-core langchain-ollama

In [4]:
import time
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableParallel,
    RunnableLambda,
    RunnableBranch,
)
from IPython.display import display, Markdown

# ============================================================
#  CONFIGURATION
# ============================================================
MODEL = "llama3.2:1b"

llm = ChatOllama(model=MODEL, temperature=0.7)

# Quick connectivity test
test = llm.invoke("Say 'ready' in one word.")
print(f"\u2705 Connected to Ollama | Model: {MODEL}")
print(f"   Response: {test.content}")

✅ Connected to Ollama | Model: llama3.2:1b
   Response: Yes.


---

## 1. The Runnable Protocol — Foundation of LCEL

Every component in LCEL implements the **Runnable protocol** — a universal interface that makes composition possible.

| Method | Description | Use Case |
|--------|-------------|----------|
| `invoke()` | Process single input synchronously | Standard request-response |
| `stream()` | Yield results incrementally | Real-time chat UIs |
| `batch()` | Process multiple inputs in parallel | Bulk processing |
| `ainvoke()` | Async invoke | High-concurrency servers |
| `astream()` | Async streaming | Async chat apps |

**Critical insight:** When you compose Runnables with `|`, the resulting chain is *itself* a Runnable. So you can `invoke()`, `stream()`, or `batch()` the entire chain.

### Input/Output Types

| Component | Input | Output |
|-----------|-------|--------|
| `ChatPromptTemplate` | `dict` (variables) | `ChatPromptValue` (messages) |
| `ChatModel` | messages | `AIMessage` |
| `StrOutputParser` | `AIMessage` | `str` |
| `JsonOutputParser` | `AIMessage` | `dict` |
| `RunnablePassthrough` | any | same as input |
| `RunnableLambda` | any | any (depends on function) |

### Experiment 1A: The Fundamental Chain Pattern

In [6]:
# The fundamental pattern: prompt | model | parser
#
# Data flow:
#   {variables} → ChatPromptTemplate → Messages → ChatModel → AIMessage → StrOutputParser → string

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise technical assistant."),
    ("human", "{question}")
])

parser = StrOutputParser()

# Build the chain with the pipe operator
chain = prompt | llm | parser

# The chain is itself a Runnable
print(f"Chain type: {type(chain).__name__}")

# invoke() — single input, full response
result = chain.invoke({"question": "What is AI"})

print(f"Result type: {type(result).__name__}")  # str, not AIMessage
display(Markdown(result))

Chain type: RunnableSequence
Result type: TextAccessor


Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, decision-making, and perception. These systems are designed to simulate human thought processes and can be found in various forms, including:

1. **Machine Learning (ML)**: AI systems that can learn from data and improve their performance over time, without being explicitly programmed.
2. **Deep Learning (DL)**: A type of ML that uses neural networks with multiple layers to analyze data and make predictions.
3. **Narrow or Weak AI**: AI systems that are designed to perform a specific task, such as facial recognition, language translation, or playing chess.

AI can be categorized into two main types:

1. **Human-Centric AI**: Designed to work alongside humans, enhancing their capabilities and productivity.
2. **Machine-Centric AI**: Designed to operate independently, making decisions and taking actions without human intervention.

AI has numerous applications across various industries, including:

1. **Virtual Assistants**: AI-powered virtual assistants, such as Siri, Google Assistant, and Alexa, that can perform tasks, answer questions, and control smart home devices.
2. **Image and Speech Recognition**: AI systems that can recognize objects, people, and speech, enabling applications like facial recognition, object detection, and voice-to-text.
3. **Predictive Analytics**: AI-powered systems that can analyze data and make predictions, helping organizations make informed decisions.
4. **Autonomous Vehicles**: AI-driven vehicles that can navigate roads, traffic, and obstacles, improving safety and efficiency.
5. **Healthcare**: AI-assisted diagnosis, personalized medicine, and patient outcomes analysis.

While AI holds great promise, it also raises important questions about:

1. **Job displacement**: Will AI replace human workers?
2. **Bias and fairness**: How can AI systems ensure fairness and equity?
3. **Security**: How can we protect AI systems from cyber threats?

As AI continues to evolve, it's essential to address these concerns and ensure that its benefits are shared by all.

### Experiment 1B: The Runnable Protocol in Action — `invoke`, `stream`, `batch`

In [7]:
# The SAME chain supports all three modes automatically

# --- invoke: single input ---
print("=" * 50)
print("invoke() — Single input, full response")
print("=" * 50)
result = chain.invoke({"question": "What is the Runnable protocol? One sentence."})
print(result)

invoke() — Single input, full response
The Runnable protocol is a programming interface provided by the Java Virtual Machine (JVM) that allows for the creation of daemon threads, also known as non-daemon threads, which run in the background without interruption.


In [8]:
# --- stream: token by token ---
print("=" * 50)
print("stream() — Tokens arrive as generated")
print("=" * 50)

for chunk in chain.stream({"question": "Name 3 LCEL primitives. Brief."}):
    print(chunk, end="", flush=True)
print()

stream() — Tokens arrive as generated
I don't have information on LCEL primitives. Can you provide more context or information about what LCEL stands for?


In [9]:
# --- batch: multiple inputs in parallel ---
print("=" * 50)
print("batch() — Process multiple inputs at once")
print("=" * 50)

questions = [
    {"question": "What is RunnablePassthrough? One sentence."},
    {"question": "What is RunnableParallel? One sentence."},
    {"question": "What is RunnableLambda? One sentence."},
]

start = time.time()
results = chain.batch(questions)
elapsed = time.time() - start

for q, r in zip(questions, results):
    print(f"Q: {q['question']}")
    print(f"A: {r}\n")

print(f"Processed {len(questions)} inputs in {elapsed:.2f}s (parallel!)")

batch() — Process multiple inputs at once
Q: What is RunnablePassthrough? One sentence.
A: RunnablePassthrough is a feature in Android that allows an activity to perform actions that do not require any user interaction, such as loading data or performing background tasks, without blocking the main execution flow of the app.

Q: What is RunnableParallel? One sentence.
A: RunnableParallel is a feature in Java that allows for parallel execution of tasks or threads, enabling concurrent execution of code with minimal synchronization overhead.

Q: What is RunnableLambda? One sentence.
A: RunnableLambda is a Java feature that allows you to create a lambda expression (a small, single-method anonymous function) that can be executed as a Runnable, enabling you to write more concise and readable code.

Processed 3 inputs in 8.83s (parallel!)


---

## 2. Extending the Chain — Adding More Components

Each additional component in the pipe adds a processing step. The chain remains a single Runnable.

```
chain = prompt | model | parser | post_processor | formatter
```

### Experiment 2A: Extending with Post-Processing

In [10]:
# Add a post-processing step using RunnableLambda
def add_disclaimer(text):
    return f"{text}\n\n--- Generated by {MODEL} via LCEL ---"

# Extended chain: prompt → model → parse → post-process
extended_chain = prompt | llm | StrOutputParser() | RunnableLambda(add_disclaimer)

result = extended_chain.invoke({"question": "What is LangChain? One sentence."})
print(result)

LangChain is a decentralized, open-source, and community-driven blockchain platform designed for the development of distributed applications and smart contracts.

--- Generated by llama3.2:1b via LCEL ---


### Experiment 2B: Chain of Chains — Composing Multiple Chains

In [11]:
# Chain 1: Generate a technical explanation
explain_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a technical writer. 2-3 sentences."),
        ("human", "Explain: {topic}")
    ])
    | llm
    | StrOutputParser()
)

# Chain 2: Simplify for a child
simplify_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Simplify this for a 10-year-old. Use an analogy. 2 sentences."),
        ("human", "{text}")
    ])
    | llm
    | StrOutputParser()
)

# Run in sequence — output of chain 1 feeds into chain 2
topic = "LCEL pipe operator"

print("Step 1 — Technical Explanation:")
print("=" * 50)
explanation = explain_chain.invoke({"topic": topic})
display(Markdown(explanation))

print("\nStep 2 — Simplified:")
print("=" * 50)
simple = simplify_chain.invoke({"text": explanation})
display(Markdown(simple))

Step 1 — Technical Explanation:


A LCEL pipe operator, also known as a Lay Pipe Line Operator or Lay Down Crew member, is a crucial personnel who is responsible for laying and supporting the installation of gas, oil, or water pipes, pipelines, and other underground infrastructure. They work in remote or hard-to-reach locations, often in harsh weather conditions, to perform routine maintenance, repairs, and upgrades on the pipeline system. The LCEL pipe operator plays a vital role in ensuring the safe and efficient operation of the pipeline network.


Step 2 — Simplified:


Imagine you're a very important person who has to lay down the big roads that people and animals use to get from one place to another. You need to make sure the roads are safe and good to use, so you spend most of your time in the middle of nowhere, in very cold or wet weather, and it's a big job!

---

## 3. LCEL Primitive 1: RunnablePassthrough — Pass Data Through

`RunnablePassthrough` passes its input through **unchanged** to the next component. Seems trivial, but it's essential for preserving data when you need it downstream.

**`RunnablePassthrough.assign()`** — keeps the original input AND adds new keys. Crucial for enriching data as it flows.

### Experiment 3A: RunnablePassthrough Basics

In [12]:
# RunnablePassthrough — input passes through unchanged
passthrough = RunnablePassthrough()

result = passthrough.invoke("Hello, I pass right through!")
print(f"Input:  'Hello, I pass right through!'")
print(f"Output: '{result}'")
print(f"Same?   {result == 'Hello, I pass right through!'}")

Input:  'Hello, I pass right through!'
Output: 'Hello, I pass right through!'
Same?   True


### Experiment 3B: RunnablePassthrough.assign() — Enrich Data

In [13]:
# assign() keeps original input AND adds new computed keys

# Simulate: given a question, add a "word_count" and "uppercased" field
enricher = RunnablePassthrough.assign(
    word_count=RunnableLambda(lambda x: len(x["question"].split())),
    uppercased=RunnableLambda(lambda x: x["question"].upper())
)

result = enricher.invoke({"question": "What is LCEL in LangChain?"})

print("Input:  {'question': 'What is LCEL in LangChain?'}")
print(f"Output: {result}")
print()
print("Original 'question' is preserved, plus new keys added!")

Input:  {'question': 'What is LCEL in LangChain?'}
Output: {'question': 'What is LCEL in LangChain?', 'word_count': 5, 'uppercased': 'WHAT IS LCEL IN LANGCHAIN?'}

Original 'question' is preserved, plus new keys added!


### Experiment 3C: assign() in a Real Chain — Adding Context

In [14]:
# Practical use: enrich input with LLM-generated context before the main prompt

# Step 1: Generate keywords from the question
keyword_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Extract 3 keywords from this question. Comma-separated, no extra text."),
        ("human", "{question}")
    ])
    | llm
    | StrOutputParser()
)

# Step 2: Use assign() to keep question AND add keywords
enriched_chain = RunnablePassthrough.assign(
    keywords=keyword_chain
)

# Step 3: Use both in the final prompt
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question. Focus on these keywords: {keywords}"),
    ("human", "{question}")
])

full_chain = enriched_chain | final_prompt | llm | StrOutputParser()

result = full_chain.invoke({"question": "How does LCEL handle streaming in chains?"})

print("Result:")
display(Markdown(result))

Result:


I'm not aware of any specific information about LCEL handling streaming in chains. My previous response was an attempt to provide a helpful answer, but I may not have the most up-to-date or accurate information. Can you please provide more context or details about LCEL and streaming in chains? This will help me better understand your question and provide a more accurate response.

---

## 4. LCEL Primitive 2: RunnableParallel — Execute Branches Simultaneously

`RunnableParallel` takes a single input and routes it to **multiple Runnables that execute in parallel**. Results are collected into a dictionary.

```
Input
  ├──→ Branch A → result_a
  ├──→ Branch B → result_b
  └──→ Branch C → result_c
  ▼
{"a": result_a, "b": result_b, "c": result_c}
```

**Dictionary shorthand:** `{"key": runnable}` is automatically treated as `RunnableParallel`.

### Experiment 4A: Basic RunnableParallel

In [15]:
# Two branches process the same input in parallel

summary_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Summarize in one sentence."),
        ("human", "{topic}")
    ])
    | llm | StrOutputParser()
)

keywords_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "List 3 keywords. Comma-separated only."),
        ("human", "{topic}")
    ])
    | llm | StrOutputParser()
)

# RunnableParallel — both execute at the same time
parallel_chain = RunnableParallel(
    summary=summary_chain,
    keywords=keywords_chain
)

start = time.time()
result = parallel_chain.invoke({"topic": "LangChain Expression Language (LCEL)"})
elapsed = time.time() - start

print(f"Completed in {elapsed:.2f}s (both branches ran in parallel!)\n")
print(f"Summary:  {result['summary']}")
print(f"Keywords: {result['keywords']}")

Completed in 6.09s (both branches ran in parallel!)

Summary:  I couldn't find any information on the LangChain Expression Language (LCEL). It's possible that it's a proprietary or internal language used by LangChain, a blockchain-based platform, or it may be a relatively unknown or emerging language. If you have more context or details about LCEL, I'd be happy to try and help further.
Keywords: Language, Compiler, Parser


### Experiment 4B: Dictionary Shorthand (Most Common Pattern)

In [16]:
# Dictionary shorthand — cleaner syntax for RunnableParallel
# This is how it's used most often in real LCEL chains

analyze_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are given a topic with its summary and keywords.\n"
     "Summary: {summary}\n"
     "Keywords: {keywords}\n"
     "Write a brief analysis (2 sentences)."),
    ("human", "Analyze this topic.")
])

# Chain: parallel branches → combine into prompt → model → parse
full_chain = (
    {"summary": summary_chain, "keywords": keywords_chain}  # dict shorthand = RunnableParallel
    | analyze_prompt
    | llm
    | StrOutputParser()
)

result = full_chain.invoke({"topic": "The Runnable protocol in LangChain"})
display(Markdown(result))

Based on the provided information, it appears that the topic is related to LangChain, a framework for building scalable and concurrent systems. However, the "Runnable protocol" is not a standard or widely recognized concept in LangChain or any other related technology, suggesting that this topic may be specific or experimental.

Without more context or information about the "Runnable protocol" in LangChain, it is difficult to provide a more detailed analysis or insights. If you could provide more information or clarify the topic, I would be happy to try and assist you further.

### Experiment 4C: RunnableParallel + RunnablePassthrough (The RAG Pattern)

This is the most important LCEL composition — used in every RAG application.

```
Input (question)
  ├──→ "Retriever" → context (documents)
  └──→ RunnablePassthrough → question (preserved)
  ▼
Prompt (combines context + question) → Model → Parser → Answer
```

In [17]:
# Simulated RAG pattern — no vector DB needed, just the LCEL structure

# Simulate a retriever with a function
knowledge_base = {
    "lcel": "LCEL is LangChain's declarative composition framework using the pipe operator.",
    "runnable": "The Runnable protocol provides invoke(), stream(), and batch() on every component.",
    "passthrough": "RunnablePassthrough forwards input unchanged, assign() adds new keys.",
    "parallel": "RunnableParallel executes multiple branches simultaneously.",
}

def fake_retriever(query: str) -> str:
    """Simulate document retrieval by keyword matching."""
    query_lower = query.lower()
    matched = [doc for key, doc in knowledge_base.items() if key in query_lower]
    return " ".join(matched) if matched else "No relevant documents found."

# RAG chain using the dictionary shorthand pattern
rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the question based ONLY on this context:\n\n{context}\n\n"
     "If the context doesn't contain the answer, say so."),
    ("human", "{question}")
])

rag_chain = (
    {"context": RunnableLambda(lambda x: fake_retriever(x["question"])),
     "question": RunnablePassthrough() | RunnableLambda(lambda x: x["question"])}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Test with questions
questions = [
    "What is LCEL?",
    "How does RunnableParallel work?",
    "What is the weather today?",  # Not in knowledge base
]

for q in questions:
    print(f"\n{'=' * 50}")
    print(f"Q: {q}")
    print(f"{'=' * 50}")
    answer = rag_chain.invoke({"question": q})
    print(f"A: {answer}")


Q: What is LCEL?
A: I don't have enough information to provide an accurate answer about LCEL. Can you please provide more context or information about what LCEL is?

Q: How does RunnableParallel work?
A: I can't answer that.

Q: What is the weather today?
A: Since there are no relevant documents provided, I don't have any information about the current weather.


---

## 5. LCEL Primitive 3: RunnableLambda — Custom Functions as Runnables

`RunnableLambda` wraps any Python function into a Runnable, making it composable in LCEL chains.

Use it for: data transformation, validation, formatting, external API calls, logging — anything that doesn't fit a built-in component.

### Experiment 5A: Basic RunnableLambda

In [18]:
# Wrap any function as a Runnable
def word_count(text: str) -> dict:
    """Count words and characters."""
    return {
        "text": text,
        "words": len(text.split()),
        "chars": len(text)
    }

counter = RunnableLambda(word_count)

# It supports the full Runnable interface
result = counter.invoke("LCEL makes chains composable and powerful")
print(f"invoke(): {result}")

# batch() works too
results = counter.batch(["Hello world", "LCEL is great", "Three word sentence"])
print(f"\nbatch(): {results}")

invoke(): {'text': 'LCEL makes chains composable and powerful', 'words': 6, 'chars': 41}

batch(): [{'text': 'Hello world', 'words': 2, 'chars': 11}, {'text': 'LCEL is great', 'words': 3, 'chars': 13}, {'text': 'Three word sentence', 'words': 3, 'chars': 19}]


### Experiment 5B: RunnableLambda in a Chain — Custom Post-Processing

In [19]:
# Custom functions as chain steps

def format_as_bullet_points(text: str) -> str:
    """Convert newline-separated text into bullet points."""
    lines = [line.strip() for line in text.strip().split("\n") if line.strip()]
    return "\n".join(f"  * {line}" for line in lines)

def add_header(text: str) -> str:
    """Add a formatted header."""
    return f"{'=' * 40}\n  LCEL CHEAT SHEET\n{'=' * 40}\n{text}\n{'=' * 40}"

# Chain: prompt → model → parse → format → header
formatted_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "List each item on its own line. No numbering."),
        ("human", "List the 5 LCEL primitives and what each does.")
    ])
    | llm
    | StrOutputParser()
    | RunnableLambda(format_as_bullet_points)
    | RunnableLambda(add_header)
)

result = formatted_chain.invoke({})
print(result)

  LCEL CHEAT SHEET
  * The 5 LCEL (Logical Concurrency Evaluation Logic) primitives are:
  * 1. Assignor
  * 2. Assignee
  * 3. Wait
  * 4. Notify
  * 5. Receive


### Experiment 5C: The @chain Decorator — Cleaner Syntax

In [20]:
from langchain_core.runnables import chain as chain_decorator

# The @chain decorator turns a function into a Runnable
# Cleaner than wrapping with RunnableLambda

@chain_decorator
def analyze_and_respond(input_dict: dict) -> str:
    """A custom Runnable that does multi-step processing."""
    question = input_dict["question"]

    # Step 1: Classify the question type
    classify_chain = (
        ChatPromptTemplate.from_messages([
            ("system", "Classify this as 'technical' or 'general'. One word only."),
            ("human", "{q}")
        ])
        | llm | StrOutputParser()
    )
    category = classify_chain.invoke({"q": question}).strip().lower()

    # Step 2: Answer based on classification
    if "technical" in category:
        style = "a precise software engineer with code examples"
    else:
        style = "a friendly teacher using simple analogies"

    answer_chain = (
        ChatPromptTemplate.from_messages([
            ("system", "You are {style}. Keep answers to 2-3 sentences."),
            ("human", "{q}")
        ])
        | llm | StrOutputParser()
    )

    answer = answer_chain.invoke({"style": style, "q": question})
    return f"[{category.upper()}] {answer}"

# Use it like any other Runnable
result = analyze_and_respond.invoke({"question": "How does the pipe operator work in LCEL?"})
print(result)

print()

result2 = analyze_and_respond.invoke({"question": "Why is the sky blue?"})
print(result2)

[I'M NOT AWARE OF ANY INFORMATION ABOUT A "PIPE OPERATOR" IN LCEL (LIKELY A FICTIONAL OR HYPOTHETICAL TERM). COULD YOU PROVIDE MORE CONTEXT OR CLARIFY WHAT LCEL STANDS FOR? I'LL DO MY BEST TO ASSIST.] Think of the pipe operator in LCEL like a conductor leading an orchestra. Just as the conductor directs the musicians to play in harmony, the pipe operator helps the air flow through the system to deliver the desired temperature. They ensure that the air passes through the coils safely and efficiently.

[TECHNICAL.] The sky appears blue because of a phenomenon called Rayleigh scattering, in which shorter (blue) wavelengths of light are scattered more than longer (red) wavelengths by the small molecules of gases in the atmosphere, such as nitrogen and oxygen. This scattering effect is more pronounced during the daytime when the sun is overhead, resulting in a blue color.


---

## 6. LCEL Primitive 4: RunnableBranch — Conditional Routing

`RunnableBranch` routes input to **different Runnables based on conditions**. Like an `if/elif/else` for chains.

```
Input
  ▼
RunnableBranch
  ├── condition_1 is True → chain_a
  ├── condition_2 is True → chain_b
  └── default → chain_c
```

### Experiment 6A: Basic Conditional Routing

In [21]:
# Different chains for different input types

technical_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a precise engineer. Use technical terminology. 2 sentences."),
        ("human", "{question}")
    ])
    | llm | StrOutputParser()
    | RunnableLambda(lambda x: f"[TECHNICAL] {x}")
)

creative_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a creative storyteller. Use metaphors and vivid language. 2 sentences."),
        ("human", "{question}")
    ])
    | llm | StrOutputParser()
    | RunnableLambda(lambda x: f"[CREATIVE] {x}")
)

general_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant. Be concise. 2 sentences."),
        ("human", "{question}")
    ])
    | llm | StrOutputParser()
    | RunnableLambda(lambda x: f"[GENERAL] {x}")
)

# Route based on keywords in the question
router = RunnableBranch(
    (lambda x: any(w in x["question"].lower() for w in ["code", "api", "function", "implement"]),
     technical_chain),
    (lambda x: any(w in x["question"].lower() for w in ["story", "imagine", "creative", "poem"]),
     creative_chain),
    general_chain  # default
)

# Test with different types of questions
test_questions = [
    "How do I implement a chain in code?",
    "Imagine LCEL as a story character.",
    "What is the weather like today?",
]

for q in test_questions:
    print(f"\n{'=' * 50}")
    print(f"Q: {q}")
    print(f"{'=' * 50}")
    result = router.invoke({"question": q})
    print(result)


Q: How do I implement a chain in code?
[TECHNICAL] To implement a linked list chain in code, you can create a static array to store the head of each chain, where each element points to the next chain, effectively forming a linked list. You can then initialize the first element of the array as the head of the first chain, and iterate through the array to create each subsequent chain, linking them together through the pointers.

Q: Imagine LCEL as a story character.
[CREATIVE] LCEL was a mystical, shimmering thread that wove through the fabric of time and space, its delicate tendrils tangling with the memories of countless civilizations, each one a thread in the ever-unraveling tapestry of the universe. As it spun, LCEL's whispers became a gentle hum, a soft lullaby that echoed through the ages, beckoning those who listened to unravel the secrets of the cosmos.

Q: What is the weather like today?
[GENERAL] I'm not capable of providing real-time weather information or predicting the weat

### Experiment 6B: LLM-Powered Routing (Classify then Route)

In [22]:
# Use the LLM itself to classify, then route to the right chain

# Step 1: Classify the question
classifier = (
    ChatPromptTemplate.from_messages([
        ("system",
         "Classify this question into exactly ONE category: technical, creative, or general.\n"
         "Respond with the category name only. One word."),
        ("human", "{question}")
    ])
    | llm | StrOutputParser()
)

# Step 2: Use assign() to keep question AND add classification
classified_chain = RunnablePassthrough.assign(
    category=classifier
)

# Step 3: Route based on classification
smart_router = RunnableBranch(
    (lambda x: "technical" in x["category"].lower(), technical_chain),
    (lambda x: "creative" in x["category"].lower(), creative_chain),
    general_chain
)

# Full pipeline: classify → route → respond
smart_chain = classified_chain | smart_router

test_qs = [
    "How does the Runnable protocol handle async execution?",
    "Write me a poem about data pipelines.",
    "What should I have for lunch?",
]

for q in test_qs:
    print(f"\n{'=' * 50}")
    print(f"Q: {q}")
    # Show classification
    classified = classified_chain.invoke({"question": q})
    print(f"Category: {classified['category'].strip()}")
    print(f"{'=' * 50}")
    result = smart_router.invoke(classified)
    print(result)


Q: How does the Runnable protocol handle async execution?
Category: technical
[TECHNICAL] The Runnable protocol, as defined in the Java 8 API, handles async execution through its `run()` method, which allows a Runnable to be executed concurrently by a thread pool, utilizing a thread pool's `Runnable` task. This method is designed to be non-blocking, enabling the execution of the Runnable to continue without blocking the main thread, whereas the `run()` method itself is thread-safe and can be used for synchronization or other concurrent tasks.

Q: Write me a poem about data pipelines.
Category: Technical
[TECHNICAL] Through data streams, we carefully navigate,
A network of flows, where data converges and waits,
From ingestion to transformation, we orchestrate the process, precise and deliberate, with data pipelines that ensure the data's integrity and accuracy, in real-time and without fail.

Q: What should I have for lunch?
Category: General.
[GENERAL] Considering you haven't specifie

---

## 7. LCEL Primitive 5: Fallbacks — Error Recovery

`.with_fallbacks()` wraps a Runnable with backup alternatives. If the primary fails, it automatically tries the next one.

```python
robust_model = primary_model.with_fallbacks([backup_model_1, backup_model_2])
```

### Experiment 7A: Fallback Chain

In [23]:
# Simulate a failing model with a function that always raises

def unreliable_model(input_text):
    """Simulates an LLM that fails."""
    raise ConnectionError("Model API is down!")

# Primary: the unreliable model
primary = RunnableLambda(unreliable_model)

# Fallback: our reliable Ollama model
fallback_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant. Be concise."),
        ("human", "{question}")
    ])
    | llm | StrOutputParser()
)

# Combine with fallback
resilient_chain = primary.with_fallbacks([fallback_chain])

# The primary will fail, but the fallback catches it
result = resilient_chain.invoke({"question": "What is LCEL?"})

print(f"Primary failed but fallback succeeded!")
print(f"Result: {result}")

Primary failed but fallback succeeded!
Result: I'm not familiar with the term "LCEL." Could you provide more context or information about what LCEL stands for or is related to? I'd be happy to try and help you find the answer.


### Experiment 7B: Retry Logic with `.with_retry()`

In [24]:
# .with_retry() adds automatic retries with exponential backoff

attempt_count = 0

def flaky_function(x):
    """Fails twice, then succeeds on the third attempt."""
    global attempt_count
    attempt_count += 1
    if attempt_count < 3:
        print(f"  Attempt {attempt_count}: FAILED")
        raise ConnectionError(f"Transient error on attempt {attempt_count}")
    print(f"  Attempt {attempt_count}: SUCCESS")
    return f"Result after {attempt_count} attempts: {x}"

# Wrap with retry logic
retry_runnable = RunnableLambda(flaky_function).with_retry(
    stop_after_attempt=5,
    wait_exponential_jitter=True
)

print("Invoking flaky function with retry...")
result = retry_runnable.invoke("hello")
print(f"\nFinal result: {result}")

Invoking flaky function with retry...
  Attempt 1: FAILED
  Attempt 2: FAILED
  Attempt 3: SUCCESS

Final result: Result after 3 attempts: hello


### Experiment 7C: Combining Retry + Fallback (Maximum Resilience)

The most robust pattern:
```python
resilient = (
    primary.with_retry(stop_after_attempt=2)
    .with_fallbacks([
        fallback.with_retry(stop_after_attempt=2)
    ])
)
```

In [25]:
# Demonstrate the pattern conceptually

# In production with multiple providers, this would look like:
# resilient_llm = (
#     ChatOpenAI(model="gpt-4o").with_retry(stop_after_attempt=2)
#     .with_fallbacks([
#         ChatAnthropic(model="claude-3-5-sonnet").with_retry(stop_after_attempt=2),
#         ChatOllama(model="llama3.2:3b")  # local fallback, no retry needed
#     ])
# )

# With our local setup — retry the same model
resilient_llm = llm.with_retry(stop_after_attempt=3)

resilient_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Be concise. One sentence."),
        ("human", "{question}")
    ])
    | resilient_llm
    | StrOutputParser()
)

result = resilient_chain.invoke({"question": "What makes LCEL resilient?"})
print(result)

LCEL (Las Cruces, New Mexico) is a resilient community due to its strong sense of community, economic diversification, and a history of natural disasters.


---

## 8. Streaming Deep Dive

When you call `.stream()` on an LCEL chain, every component streams its output to the next as tokens become available.

| Component | Streaming Behavior |
|-----------|-----------------------|
| Chat Model | Yields tokens as generated |
| StrOutputParser | Passes through each token immediately |
| JsonOutputParser | Accumulates until valid JSON, then yields |
| RunnablePassthrough | Passes through immediately |
| RunnableLambda | Depends on function (generators stream) |
| Retriever | Returns complete results (no partial streaming) |

### Experiment 8A: Streaming Through a Chain — Token Timing

In [26]:
# Compare: invoke (wait for all) vs stream (token by token)

chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant."),
        ("human", "{question}")
    ])
    | llm | StrOutputParser()
)

question = {"question": "List 3 benefits of Langchain Expression Language in langchain. One sentence each."}

# --- invoke: wait for complete response ---
print("=" * 50)
print("invoke() — User waits for full response")
print("=" * 50)
start = time.time()
result = chain.invoke(question)
elapsed = time.time() - start
print(result)
print(f"\nTotal wait: {elapsed:.2f}s")

invoke() — User waits for full response
I can provide you with general information about Langchain Expression Language. Here are three benefits:

1. **Efficient memory management**: Langchain Expression Language allows for seamless memory management, enabling developers to focus on the logic of their application rather than dealing with memory issues.
2. **Highly parallelizable**: The language's design enables high parallelism, making it well-suited for distributed computing and parallel processing tasks, such as machine learning and data science.
3. **Flexible and extensible**: Langchain Expression Language's modular design and extensibility features make it easy to integrate with various frameworks and libraries, allowing developers to build custom components and workflows.

Total wait: 11.91s


In [27]:
# --- stream: tokens arrive immediately ---
print("=" * 50)
print("stream() — Tokens arrive as generated")
print("=" * 50)
start = time.time()
first_token_time = None
token_count = 0

for chunk in chain.stream(question):
    if first_token_time is None:
        first_token_time = time.time() - start
    print(chunk, end="", flush=True)
    token_count += 1

elapsed = time.time() - start
print(f"\n\nTime to first token: {first_token_time:.2f}s")
print(f"Total time: {elapsed:.2f}s")
print(f"Chunks received: {token_count}")
print("\nUsers perceive streaming as faster even when total time is similar!")

stream() — Tokens arrive as generated
I can provide you with information on Langchain Expression Language (LEL) benefits. 

Here are three benefits of LEL in Langchain:

1. **Improved performance**: LEL allows for more efficient and scalable computations by leveraging the strengths of the Rust programming language.
2. **Faster development**: LEL provides a more concise and expressive way of writing programs, which can speed up the development process for developers.
3. **Increased security**: By using a secure and private encryption mechanism, LEL protects sensitive data and prevents unauthorized access to Langchain's infrastructure.

Time to first token: 0.12s
Total time: 9.89s
Chunks received: 116

Users perceive streaming as faster even when total time is similar!


---

## 9. Batch Processing Deep Dive

`.batch()` processes multiple inputs in **parallel**. Configurable with `max_concurrency`.

| Parameter | Default | Description |
|-----------|---------|-------------|
| `max_concurrency` | None (unlimited) | Max parallel executions |
| `return_exceptions` | False | Return exceptions instead of raising |

### Experiment 9A: Batch with Timing Comparison

In [27]:
chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Answer in exactly one sentence."),
        ("human", "{question}")
    ])
    | llm | StrOutputParser()
)

questions = [
    {"question": "What is LCEL?"},
    {"question": "What is a Runnable?"},
    {"question": "What is RunnablePassthrough?"},
    {"question": "What is RunnableParallel?"},
]

# --- Sequential (invoke one by one) ---
print("Sequential (invoke one by one):")
start = time.time()
sequential_results = [chain.invoke(q) for q in questions]
seq_time = time.time() - start
print(f"  Time: {seq_time:.2f}s")

# --- Batch (all at once) ---
print("\nBatch (parallel):")
start = time.time()
batch_results = chain.batch(questions)
batch_time = time.time() - start
print(f"  Time: {batch_time:.2f}s")

print(f"\nSpeedup: {seq_time/batch_time:.1f}x faster with batch!")

print("\nResults:")
for q, r in zip(questions, batch_results):
    print(f"  Q: {q['question']}")
    print(f"  A: {r}\n")

Sequential (invoke one by one):
  Time: 23.07s

Batch (parallel):
  Time: 27.05s

Speedup: 0.9x faster with batch!

Results:
  Q: What is LCEL?
  A: LCE (Local Contrastive Encoding) is a supervised learning method used to optimize the embedding space of a neural network, such as a sentence transformer model. It is particularly useful for tasks like text classification and clustering.

In traditional contrastive learning methods, similar samples are paired together in the training process to encourage their embeddings to be close, while dissimilar samples are paired together to discourage their embeddings from being too similar. However, this approach can lead to overfitting when the dataset size is limited or when there are many classes with few samples.

LCE addresses these challenges by introducing a new loss function that focuses on local contrastive learning. The idea is to find a balance between encouraging similarity and discouraging similarity, but in a more targeted way.

In LC

In [28]:
# Batch with max_concurrency — respect rate limits
print("Batch with max_concurrency=2:")
start = time.time()
results = chain.batch(questions, config={"max_concurrency": 2})
elapsed = time.time() - start
print(f"  Time: {elapsed:.2f}s (limited to 2 concurrent requests)")
print(f"  Results: {len(results)} answers")

Batch with max_concurrency=2:
  Time: 29.38s (limited to 2 concurrent requests)
  Results: 4 answers


---

## 10. Advanced Pattern: Sequential Processing with Context Accumulation

Build up context progressively — each step has access to everything produced by previous steps.

```
Input → Step 1 (summary) → Step 2 (key points using summary) → Step 3 (report using all above)
```

### Experiment 10A: Multi-Step Analysis with assign()

In [33]:
# Progressive context accumulation using assign()

# Step 1: Summarize the topic
summarize = (
    ChatPromptTemplate.from_messages([
        ("system", "Summarize in 1-2 sentences."),
        ("human", "{topic}")
    ])
    | llm | StrOutputParser()
)

# Step 2: Extract key points (has access to topic + summary)
extract_points = (
    ChatPromptTemplate.from_messages([
        ("system",
         "Given this summary: {summary}\n\n"
         "List 3 key points about the original topic. One line each."),
        ("human", "{topic}")
    ])
    | llm | StrOutputParser()
)

# Step 3: Generate a verdict (has access to topic + summary + key_points)
verdict = (
    ChatPromptTemplate.from_messages([
        ("system",
         "Based on:\nSummary: {summary}\nKey Points: {key_points}\n\n"
         "Give a one-sentence verdict on the topic's importance."),
        ("human", "{topic}")
    ])
    | llm | StrOutputParser()
)

# Chain with progressive context accumulation
analysis_chain = (
    RunnablePassthrough.assign(summary=summarize)         # adds 'summary'
    | RunnablePassthrough.assign(key_points=extract_points)  # adds 'key_points'
    | RunnablePassthrough.assign(verdict=verdict)            # adds 'verdict'
)

result = analysis_chain.invoke({"topic": "LCEL (LangChain Expression Language)"})

print("PROGRESSIVE ANALYSIS")
print("=" * 50)
print(f"\nTopic: {result['topic']}")
print(f"\nSummary:\n  {result['summary']}")
print(f"\nKey Points:\n  {result['key_points']}")
print(f"\nVerdict:\n  {result['verdict']}")

PROGRESSIVE ANALYSIS

Topic: LCEL (LangChain Expression Language)

Summary:
  LCLE (LangChain Expression Language, pronounced "lecle") is a programming language designed specifically for natural language processing (NLP) tasks, particularly those involving text generation, summarization, and text manipulation.

**Overview**

LCLE is an extension of the LangChain ecosystem, which provides a set of libraries and tools for building NLP applications. LCLE adds support for defining custom expression languages that can be used to manipulate and generate text.

**Key Features**

1. **Expression Language**: LCLE introduces a new programming language called Expression Language (EXL) that allows you to define custom expressions using a simple syntax.
2. **Template-based Text Generation**: LCLE provides a template-based approach for generating text, which enables you to create complex templates and manipulate them dynamically.
3. **Text Manipulation**: LCLE includes a range of text manipulation t

---

## 11. LCEL Best Practices

| Practice | Description |
|----------|-------------|
| Keep chains simple | Start with `prompt \| model \| parser`, add complexity only when needed |
| Name your chains | Use `.with_config(run_name="name")` for easier debugging |
| Use `assign()` for context | Cleaner than manual dictionary construction |
| Prefer dict shorthand | `{"key": runnable}` over explicit `RunnableParallel` |
| Always add fallbacks | LLM APIs fail — plan for it with `.with_fallbacks()` |
| Use retry with backoff | `.with_retry()` handles transient errors gracefully |
| Set `max_concurrency` | Respect API rate limits when using `.batch()` |
| Stream for user-facing apps | Users perceive streaming as faster |
| Test with `invoke()` first | Debug synchronous execution before adding streaming |

### LCEL vs Imperative Code

| Aspect | LCEL (Declarative) | Imperative Code |
|--------|--------------------|-----------------|
| Readability | Excellent for linear/parallel flows | Better for complex branching |
| Streaming | Automatic, first-class | Manual implementation |
| Async | Built-in for every chain | Must rewrite with async/await |
| Batching | One method call | Custom threading |
| Tracing | Automatic (LangSmith) | Manual logging |
| Error handling | Declarative retry/fallback | Try/except blocks |
| Best for | 90% of LLM pipelines | Highly dynamic workflows |

> **Recommendation:** Use LCEL for 90% of your chains. Fall back to imperative code only for highly dynamic workflows — or consider **LangGraph** for those.

---

## 12. Sandbox — Try It Yourself!

In [ ]:
# ============================================================
#  SANDBOX - Edit and re-run!
# ============================================================

# Build your own LCEL chain
my_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Be concise."),
    ("human", "{question}")
])

my_chain = my_prompt | llm | StrOutputParser()

# Try different modes
mode = "stream"  # Options: "invoke", "stream", "batch"

# ============================================================

if mode == "invoke":
    print("[INVOKE]\n")
    result = my_chain.invoke({"question": "What is the most important LCEL primitive and why?"})
    display(Markdown(result))

elif mode == "stream":
    print("[STREAM]\n")
    for chunk in my_chain.stream({"question": "What is the most important LCEL primitive and why?"}):
        print(chunk, end="", flush=True)
    print()

elif mode == "batch":
    print("[BATCH]\n")
    results = my_chain.batch([
        {"question": "What is RunnablePassthrough?"},
        {"question": "What is RunnableBranch?"},
        {"question": "What is RunnableLambda?"},
    ])
    for r in results:
        print(f"  {r}\n")

---

## Key Takeaways

| Concept | What to Remember |
|---------|------------------|
| **LCEL** | Declarative composition with the pipe `\|` operator |
| **Runnable Protocol** | Universal interface: `invoke()`, `stream()`, `batch()` on every component |
| **Fundamental Pattern** | `prompt \| model \| parser` — the atom of every chain |
| **RunnablePassthrough** | Forwards data unchanged; `assign()` enriches with new keys |
| **RunnableParallel** | Executes branches simultaneously; dict shorthand `{"key": runnable}` |
| **RunnableLambda** | Wraps any Python function as a composable Runnable |
| **@chain decorator** | Cleaner syntax for custom Runnables |
| **RunnableBranch** | Conditional routing — if/elif/else for chains |
| **with_fallbacks()** | Automatic failover to backup models/chains |
| **with_retry()** | Automatic retries with exponential backoff |
| **Streaming** | Automatic through the entire chain — no extra code needed |
| **Batching** | `.batch()` with `max_concurrency` for parallel + rate-limited processing |
| **RAG Pattern** | `{"context": retriever, "question": passthrough} \| prompt \| model \| parser` |
| **Context Accumulation** | Chain `assign()` calls to progressively build up data |